# Refiner visual transform

Visualize the frozen VSR contexts before and after applying the visual-only refiner transform.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..")
    if os.path.basename(os.getcwd()) == "notebooks"
    else os.getcwd()
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from srcs.datasets.vicocktail import load_vicocktail
from srcs.nets.backend.nets_utils import make_non_pad_mask
from srcs.nets.backend.refiner.transform import RefinerTransform, VISUAL_KEYS
from srcs.nets.e2e import get_model
from srcs.spm.spm_train import ensure_unigram
from srcs.spm.text_transofm import TextTransform
from srcs.trainer.utils import create_dataloader, load_config, move_batch, set_seed

CONFIG_PATH = os.path.join(PROJECT_ROOT, "config.yaml")
CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "finetune_vsr_12",
    "final",
    "model.safetensors",
)
SAMPLE_COUNT = 3
TEST_FRACTION = 0.01

config = load_config(CONFIG_PATH)
seed = config["training"]["seed"]
set_seed(seed)

dataset = load_vicocktail(
    test_fraction=TEST_FRACTION,
    splits=("test",),
    seed=seed,
)["test"]
model_path, units_path = ensure_unigram()
text_transform = TextTransform(model_path, units_path)
dataloader = create_dataloader(
    dataset,
    text_transform,
    "test",
    config["evaluation"],
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model = get_model(
    "auto-vsr",
    text_transform.vocab_size,
    checkpoint=CHECKPOINT_PATH,
).to(device)
base_model.eval()

transform_config = config["training"]["transform"].copy()
transform_config["probability"] = 1.0
transform = RefinerTransform(**transform_config).train()

print(f"Device: {device}")
print(f"Samples loaded: {len(dataset)}")
print(f"Transform: {transform_config}")

In [ ]:
batch = move_batch(next(iter(dataloader)), device)
amp_enabled = config["evaluation"].get("amp", True) and device.type == "cuda"

with torch.inference_mode():
    with torch.amp.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=amp_enabled,
    ):
        logits, visual_contexts = base_model.get_contexts(
            batch["videos"], batch["video_lengths"]
        )

original_logits = logits.clone()
set_seed(seed)
transformed_contexts = transform(visual_contexts)
sample_count = min(SAMPLE_COUNT, batch["videos"].size(0))

print(f"Logits: {tuple(logits.shape)}")
for key in VISUAL_KEYS:
    print(
        f"{key}: {tuple(visual_contexts[key].shape)} -> "
        f"{tuple(transformed_contexts[key].shape)}"
    )

In [ ]:
valid_mask = make_non_pad_mask(visual_contexts["input_lengths"]).to(device)
changed_masks = {
    key: (visual_contexts[key] - transformed_contexts[key])
    .abs()
    .sum(dim=-1)
    .gt(0)
    & valid_mask
    for key in VISUAL_KEYS
}

reference_mask = changed_masks["visual_features"]
assert all(
    torch.equal(reference_mask, changed_masks[key])
    for key in VISUAL_KEYS[1:]
)
assert torch.equal(logits, original_logits)

for index in range(sample_count):
    length = int(visual_contexts["input_lengths"][index])
    masked_frames = reference_mask[index, :length].nonzero(as_tuple=True)[0]
    label_length = int(batch["label_lengths"][index])
    reference = text_transform.decode(batch["labels"][index, :label_length])

    print(f"Sample {index + 1}: {reference}")
    print(f"Length: {length}")
    print(f"Masked frames: {masked_frames.detach().cpu().tolist()}")
    print()

In [ ]:
labels = {
    "visual_features": "F0",
    "h2_features": "H2",
    "h4_features": "H4",
}
colors = {
    "visual_features": "tab:blue",
    "h2_features": "tab:orange",
    "h4_features": "tab:green",
}

figure, axes = plt.subplots(
    sample_count,
    1,
    figsize=(15, 3.5 * sample_count),
    squeeze=False,
)

for index, axis in enumerate(axes[:, 0]):
    length = int(visual_contexts["input_lengths"][index])
    frames = torch.arange(length).cpu()

    for key in VISUAL_KEYS:
        original_norm = visual_contexts[key][index, :length].float().norm(dim=-1)
        transformed_norm = (
            transformed_contexts[key][index, :length].float().norm(dim=-1)
        )
        scale = original_norm.max().clamp_min(1e-6)

        axis.plot(
            frames,
            (original_norm / scale).cpu(),
            color=colors[key],
            alpha=0.25,
            linestyle="--",
        )
        axis.plot(
            frames,
            (transformed_norm / scale).cpu(),
            color=colors[key],
            label=labels[key],
        )

    masked_frames = reference_mask[index, :length].nonzero(as_tuple=True)[0].cpu()
    for frame in masked_frames.tolist():
        axis.axvspan(frame - 0.5, frame + 0.5, color="red", alpha=0.12)

    axis.set_title(f"Sample {index + 1} — {len(masked_frames)} masked frames")
    axis.set_xlabel("Frame")
    axis.set_ylabel("Normalized feature norm")
    axis.set_ylim(-0.05, 1.1)
    axis.grid(alpha=0.2)
    axis.legend(loc="upper right")

figure.suptitle(
    "Refiner visual contexts: dashed = original, solid = transformed",
    fontsize=14,
)
figure.tight_layout()
plt.show()